## Scraping AWOIAF Character Names
Scrapes the full character list from the A Wiki of Ice and Fire and stores results in a CSV.

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

### Fetch the page

In [2]:
URL = 'https://awoiaf.westeros.org/index.php/List_of_characters'

# Full browser User-Agent required — the server returns 403 for generic bot strings
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}
page = requests.get(URL, headers=headers)

soup = BeautifulSoup(page.content, 'html.parser')
print(f'Status code: {page.status_code}')

Status code: 200


### Extract character names
The page is a MediaWiki article. Character names are listed as `<li>` items inside the main content div (`mw-parser-output`), each wrapped in an `<a>` tag.

In [3]:
# Scope to the main article body to avoid nav/footer links
content = soup.find('div', class_='mw-parser-output')

characters = []

for li in content.find_all('li'):
    a = li.find('a')  # First <a> in each li is the character link
    if a and a.get('href', '').startswith('/index.php/'):  # Skip TOC anchor links (#A, #B ...)
        name = a.get_text(strip=True)
        link = a.get('href')
        if name:
            characters.append({'name': name, 'ID': link})

print(f'Found {len(characters)} characters')
print(characters[:10])  # Preview first 10

Found 3698 characters
[{'name': 'A certain man', 'ID': '/index.php/A_certain_man'}, {'name': 'Abelar Hightower', 'ID': '/index.php/Abelar_Hightower'}, {'name': 'Abelon', 'ID': '/index.php/Abelon'}, {'name': 'Addam of Duskendale', 'ID': '/index.php/Addam_of_Duskendale'}, {'name': 'Addam Frey', 'ID': '/index.php/Addam_Frey'}, {'name': 'Addam Hightower', 'ID': '/index.php/Addam_Hightower'}, {'name': 'Addam Marbrand', 'ID': '/index.php/Addam_Marbrand'}, {'name': 'Addam Osgrey', 'ID': '/index.php/Addam_Osgrey'}, {'name': 'Addam Rivers', 'ID': '/index.php/Addam_Rivers'}, {'name': 'Addam Velaryon', 'ID': '/index.php/Addam_Velaryon'}]


### Save to CSV

In [4]:
df = pd.DataFrame(characters)
df.to_csv('characters.csv', index=False)

print(f'Saved {len(df)} characters to characters.csv')
df.head(10)

Saved 3698 characters to characters.csv


,name,ID
0,A certain man,/index.php/A_certain_man
1,Abelar Hightower,/index.php/Abelar_Hightower
2,Abelon,/index.php/Abelon
3,Addam of Duskendale,/index.php/Addam_of_Duskendale
4,Addam Frey,/index.php/Addam_Frey
5,Addam Hightower,/index.php/Addam_Hightower
6,Addam Marbrand,/index.php/Addam_Marbrand
7,Addam Osgrey,/index.php/Addam_Osgrey
8,Addam Rivers,/index.php/Addam_Rivers
9,Addam Velaryon,/index.php/Addam_Velaryon


---
## Part 2 — Enrich each character page
For every character we scrape:
- **Infobox** → `father`, `mother`, `spouse`, `lover`, `issue`, `family`, `allegiance`
- **History / narrative sections** → `affiliated` (links that resolve to a known character ID)

In [5]:
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the character list we already scraped
df = pd.read_csv('characters.csv')

# Build a set of known character IDs for fast lookup (used to filter affiliated links)
known_ids = set(df['ID'].tolist())

BASE = 'https://awoiaf.westeros.org'

# Shared session with connection pooling — reuses TCP connections across threads
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
})

INFOBOX_MAP = {
    'father':      'father',
    'mother':      'mother',
    'mothers':     'mother',
    'spouse':      'spouse',
    'spouses':     'spouse',
    'lover':       'lover',
    'lovers':      'lover',
    'consorts':    'lover',
    'issue':       'issue',
    'family':      'family',
    'allegiance':  'allegiance',
    'allegiances': 'allegiance',
}

SKIP_SECTIONS = {
    'references', 'notes', 'external links', 'see also',
    'quotes', 'behind the scenes', 'gallery',
}

def scrape_character(char_id):
    result = {col: '' for col in ['father', 'mother', 'spouse', 'lover', 'issue', 'family', 'allegiance', 'affiliated']}
    try:
        resp = session.get(BASE + char_id, timeout=15)
        if resp.status_code != 200:
            return char_id, result
        soup = BeautifulSoup(resp.content, 'html.parser')
    except Exception:
        return char_id, result

    # ── Infobox ──────────────────────────────────────────────────────────────
    infobox = soup.find('table', class_='infobox')
    if infobox:
        for row in infobox.find_all('tr'):
            th = row.find('th')
            td = row.find('td')
            if not (th and td):
                continue
            field = th.get_text(strip=True).lower()
            col = INFOBOX_MAP.get(field)
            if col is None:
                continue
            links = [a.get('href') for a in td.find_all('a') if a.get('href', '').startswith('/index.php/')]
            if links:
                result[col] = ';'.join(links)
            else:
                text = td.get_text(' ', strip=True)
                if text:
                    result[col] = text

    # ── Affiliated (narrative sections, filtered to known characters) ─────────
    content = soup.find('div', class_='mw-parser-output')
    if content:
        affiliated = []
        skip = False
        for elem in content.children:
            tag = getattr(elem, 'name', None)
            if tag == 'h2':
                skip = any(s in elem.get_text(strip=True).lower() for s in SKIP_SECTIONS)
            if skip:
                continue
            if tag in ('p', 'ul', 'ol', 'div', 'h3', 'h4'):
                for a in elem.find_all('a'):
                    href = a.get('href', '')
                    if href in known_ids and href not in affiliated:
                        affiliated.append(href)
        result['affiliated'] = ';'.join(affiliated)

    return char_id, result

print(f'Defined scraper. {len(df)} characters to enrich.')

Defined scraper. 3698 characters to enrich.


In [6]:
# 10 concurrent workers — ~10x faster than sequential
# Results come back out-of-order; we re-sort by the original index at the end

WORKERS = 10
CHECKPOINT_EVERY = 500

id_to_row = {row.ID: {'name': row.name, 'ID': row.ID} for row in df.itertuples()}
results = {}  # char_id -> result dict

futures = {}
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for char_id in df['ID']:
        futures[executor.submit(scrape_character, char_id)] = char_id

    for i, future in enumerate(tqdm(as_completed(futures), total=len(futures), desc='Scraping')):
        char_id, data = future.result()
        results[char_id] = data

        if (i + 1) % CHECKPOINT_EVERY == 0:
            checkpoint_rows = [{**id_to_row[cid], **results[cid]} for cid in results]
            pd.DataFrame(checkpoint_rows).to_csv('characters_enriched.csv', index=False)

# Rebuild in original order
rows = [{**id_to_row[cid], **results[cid]} for cid in df['ID']]
enriched_df = pd.DataFrame(rows)
enriched_df.to_csv('characters_enriched.csv', index=False)
print(f'Done. {len(enriched_df)} rows, {len(enriched_df.columns)} columns.')
enriched_df.head(10)

Scraping:   0%|          | 0/3698 [00:00<?, ?it/s]

Scraping:   0%|          | 3/3698 [00:00<02:20, 26.30it/s]

Scraping:   0%|          | 12/3698 [00:00<01:08, 54.18it/s]

Scraping:   1%|          | 21/3698 [00:00<01:02, 58.73it/s]

Scraping:   1%|          | 27/3698 [00:00<01:40, 36.62it/s]

...

Scraping: 100%|█████████▉| 3689/3698 [00:52<00:00, 70.14it/s]

Scraping: 100%|██████████| 3698/3698 [00:52<00:00, 56.67it/s]

Scraping: 100%|██████████| 3698/3698 [00:52<00:00, 70.21it/s]

Done. 3698 rows, 10 columns.


,name,ID,father,mother,spouse,lover,issue,family,allegiance,affiliated
0,A certain man,/index.php/A_certain_man,,,,,,,,/index.php/Varys;/index.php/Tyrion_Lannister;/...
1,Abelar Hightower,/index.php/Abelar_Hightower,,,,,,,/index.php/House_Hightower,/index.php/Daeron_II_Targaryen;/index.php/Dunc...
2,Abelon,/index.php/Abelon,,,,,,,/index.php/Citadel,
3,Addam of Duskendale,/index.php/Addam_of_Duskendale,,,,,,,,
4,Addam Frey,/index.php/Addam_Frey,,,,,,,/index.php/House_Frey,/index.php/Aerys_I_Targaryen;/index.php/Lord_F...
5,Addam Hightower,/index.php/Addam_Hightower,/index.php/Manfred_Hightower_(Aegon%27s_Conquest),,Unknown,,/index.php/Manfred_Hightower_(son_of_Addam);/i...,,/index.php/House_Hightower;/index.php/House_Ty...,/index.php/Aegon_I_Targaryen;/index.php/Garmon...
6,Addam Marbrand,/index.php/Addam_Marbrand,/index.php/Damon_Marbrand,,,,,,/index.php/House_Marbrand;/index.php/House_Lan...,/index.php/Damon_Marbrand;/index.php/Tywin_Lan...
7,Addam Osgrey,/index.php/Addam_Osgrey,/index.php/Eustace_Osgrey,,,,,,/index.php/House_Osgrey,/index.php/Daeron_II_Targaryen;/index.php/Eust...
8,Addam Rivers,/index.php/Addam_Rivers,,,,,,,,/index.php/Arlan_III_Durrandon
9,Addam Velaryon,/index.php/Addam_Velaryon,,/index.php/Marilda_of_Hull,,,,,/index.php/House_Velaryon;/index.php/House_Tar...,/index.php/Alyn_Velaryon;/index.php/Marilda_of...
